# Grid Topology Diff: 2025 vs. Exogenous (NEP/TYNDP) 2030 vs. Optimal 2030

Builds `stochastic_grid_scenarios` CSVs from **real** grid states instead of
arbitrary capacity reductions:

- **2025 state**: `..._2025_final.nc`, the first horizon in this myopic chain -
  existing/committed grid, not itself derived from any prior PyPSA solve.
- **exogenous 2030 state**: existing infrastructure plus every committed
  (NEP/TYNDP/manual) transmission project whose own `build_year` is `<= 2030` -
  *no endogenous optimization anywhere in the chain*. Computed directly from
  the pristine `2025_final` network (see "Exogenous-only scenarios" below for
  why - it is **not** the same thing as `2030_final`, which already bakes in
  whatever the 2025 solve itself endogenously decided to build).
- **optimal 2030 state**: `..._2030.nc`, the 2030 postnetwork - what an
  unconstrained (grid-extendable) solve actually builds on top of that.

AC lines / DC links carry a `build_year` attribute that survives clustering
(`0` = pre-existing baseline, non-zero = a specific transmission project's
commissioning year, see `transmission_projects` config and
`scripts/build_transmission_projects.py`), which is what makes this possible.
A capacity diff between two networks is attributed to:

- **planned (NEP/TYNDP)**: `build_year` falls strictly between the two
  networks' planning horizons - a committed project came online.
- **endogenously optimized**: `build_year` is `0` or at/before the earlier
  horizon - the change is the solver's own (possibly brownfield-carried)
  capacity choice, not a specific committed project.
- **unclear**: doesn't fit either bucket cleanly (e.g. a project whose
  `build_year` is later than *both* horizons being compared).

In [ ]:
import os
# Must be set before geopandas/fiona/cartopy import: a stray global PROJ_DATA
# (e.g. from an unrelated mambaforge install) pointing at an older proj.db
# breaks GDAL's own proj lookups ("DATABASE.LAYOUT.VERSION.MINOR" mismatch),
# even though pyproj itself is pointed at the right one below.
os.environ["PROJ_DATA"] = "/home/julian-geis/repos/pypsa-de/.pixi/envs/default/share/proj"
os.environ.pop("PROJ_LIB", None)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
for logger in ["pypsa", "linopy", "fiona"]:
    logging.getLogger(logger).setLevel(logging.ERROR)

from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import pypsa
import pyproj
pyproj.datadir.set_data_dir("/home/julian-geis/repos/pypsa-de/.pixi/envs/default/share/proj")

plt.rcParams["figure.dpi"] = 110
REPO = Path("/home/julian-geis/repos/pypsa-de")
REGIONS = gpd.read_file(REPO / "resources/regions_onshore_base_s_27.geojson").set_index("name")

## Config

`SCOPE` controls which branches enter every diff/plot/export below:

- `"whole_network"` - every AC line / DC link in the model, including
  connections with nothing to do with Germany (e.g. FR-ES).
- `"de_internal"` - only lines/links with both ends in Germany.
- `"de_and_interconnectors"` **(default)** - DE-internal plus DE-neighbour
  cross-border connections; excludes purely foreign branches.

In [ ]:
SCOPE = "de_and_interconnectors"  # "whole_network" | "de_internal" | "de_and_interconnectors"

CLUSTERS = "27"
NOM_ATTR = {"Line": "s_nom", "Link": "p_nom"}
WIDTH_SCALE_MW = 6000  # MW per unit of linewidth, shared by every map and legend below

## Functions

In [ ]:
def load_network(path):
    '''Quiet pypsa.Network load.'''
    logging.getLogger("pypsa").setLevel(logging.CRITICAL)
    return pypsa.Network(str(path))


def temporal_resolution_label(n):
    '''e.g. "126sn-70h": number of representative snapshots and their modal
    weight (hours each snapshot stands for) - self-describing regardless of
    what segmentation scheme produced them, unlike reading a config field
    that may not exist on an arbitrary network's `n.meta`.'''
    modal_h = int(n.snapshot_weightings.generators.mode().iloc[0])
    return f"{len(n.snapshots)}sn-{modal_h}h"


def _branches(n, component):
    return n.lines if component == "Line" else n.links.query("carrier == 'DC'")


def branch_table(n, component, horizon):
    '''One row per branch: effective capacity (opt if solved, else nom), length, build_year, endpoints.

    `horizon` is the planning year this network snapshot represents.

    Whether the network is solved is decided once, network-wide (any opt value
    > 0 at all), not per branch: PyPSA-DE models a bidirectional DC link as two
    separate directional Link objects (e.g. "DC25" / "DC25-reversed"), and it's
    entirely legitimate for the optimizer to build zero capacity on one
    direction. A per-branch "opt > 0 else nom" fallback would silently
    overwrite that real zero with the pre-solve floor capacity.

    For an *unsolved* network, build-year gating (floor to 0 before
    `build_year`) is applied to Links only, never to Lines - traced directly
    against `scripts/pypsa-de/modify_prenetwork.py`'s
    `enforce_transmission_project_build_years`, the only place in the
    pipeline that actually forces pre-build capacity to 0 (`p_nom_min =
    p_nom_max = 0` for `build_year > current_year`), and it filters to
    `carrier == "DC"` explicitly. Lines have no equivalent: a line-upgrade
    project's row (`get_upgraded_lines` in `build_transmission_projects.py`)
    inherits its floor from the *existing* line's own current capacity, which
    is genuinely available from year one - `build_year` there is metadata
    identifying which NEP/TYNDP/manual measure it traces to, not an
    investment-timing constraint. Confirmed empirically: every AC line with a
    future `build_year` in this run was already at full floor capacity in the
    actual 2025 solve, regardless of whether that `build_year` was 2025 or
    2037. Gating Lines by `build_year` anyway - the previous version of this
    function did - silently zeroed out capacity the model itself never gates.
    '''
    nom = NOM_ATTR[component]
    df = _branches(n, component)
    opt = df[f"{nom}_opt"]
    if (opt > 0).any():
        capacity = opt
    elif component == "Link":
        not_yet_due = df.build_year > horizon
        capacity = df[nom].where(~not_yet_due, 0.0)
    else:
        capacity = df[nom]
    return pd.DataFrame(
        {
            "capacity_mw": capacity,
            "length_km": df["length"],
            "build_year": df["build_year"],
            "bus0": df["bus0"],
            "bus1": df["bus1"],
            "country0": n.buses.loc[df.bus0, "country"].to_numpy(),
            "country1": n.buses.loc[df.bus1, "country"].to_numpy(),
        },
        index=df.index,
    )


def scope_mask(table, scope=SCOPE):
    de0, de1 = table.country0 == "DE", table.country1 == "DE"
    if scope == "de_internal":
        return de0 & de1
    if scope == "de_and_interconnectors":
        return de0 | de1
    if scope == "whole_network":
        return pd.Series(True, index=table.index)
    raise ValueError(f"unknown SCOPE {scope!r}")

### Exogenous-only scenarios

`exogenous_scenario` answers "what does the grid look like at year Y if
*only* committed (NEP/TYNDP/manual) projects are built, with zero endogenous
expansion anywhere" - by reusing `branch_table`'s own build-year gating, but
applied to `pristine`, not to whichever year's own prenetwork.

This distinction matters: `2030_final`'s "existing" (`build_year == 0`)
branches are **not** pristine - PyPSA-DE's brownfield carryover overwrites
their capacity value with the *previous* year's `_opt` in place (label stays
`0`, value doesn't). Verified directly on this run: every one of the 38
`build_year == 0` AC lines differs in `s_nom` between `2025_final` and
`2030_final`. So computing "exogenous 2030" from `2030_final` would silently
re-include whatever the 2025 solve endogenously built. `2025_final` doesn't
have this problem - it's the *first* horizon in the myopic chain, built
directly from real grid data plus committed projects, never itself
brownfield-updated from a prior PyPSA solve - so it's the correct universal
reference for *any* target year, not just 2025.

**Build-year gating only applies to Links, never to Lines** - traced against
the actual pipeline rather than assumed. `scripts/pypsa-de/modify_prenetwork.py`'s
`enforce_transmission_project_build_years` is the *only* place that forces
pre-build capacity to 0 (`p_nom_min = p_nom_max = 0` for `build_year >
current_year`), and it filters explicitly to `carrier == "DC"`. A line-upgrade
project's row (`get_upgraded_lines` in `scripts/build_transmission_projects.py`)
instead inherits its floor from the *existing* line's own current capacity -
genuinely available from year one - with `build_year` surviving only as
metadata identifying which NEP/TYNDP/manual measure it traces to, not as an
investment-timing constraint. Confirmed empirically: every AC line with a
future `build_year` in this run was already at full floor capacity in the
actual 2025 solve, regardless of whether `build_year` was 2025 or 2037.

Practical consequence: `exogenous_scenario` for **Lines** is horizon-invariant
- it always returns `pristine`'s own capacity, since nothing about it changes
with the target year. Only **Links** show real year-dependent committed-project
effects. This isn't a simplification for this notebook specifically - it's an
accurate reflection of how the underlying model actually treats the two
component types differently.

In [ ]:
def exogenous_scenario(pristine, component, target_year):
    '''"No endogenous optimization at all" grid state at `target_year`.'''
    return branch_table(pristine, component, target_year)

In [ ]:
def diff_table(table_a, table_b, horizon_a, horizon_b):
    '''Per-branch capacity delta (b - a) with a planned/endogenous/unclear classification.'''
    idx = table_a.index.union(table_b.index)
    a, b = table_a.reindex(idx), table_b.reindex(idx)

    delta = b.capacity_mw.fillna(0) - a.capacity_mw.fillna(0)
    build_year = b.build_year.fillna(a.build_year).fillna(0)

    def classify(by, d):
        if abs(d) < 1e-6:
            return "no change"
        if horizon_a < by <= horizon_b:
            return "planned (NEP/TYNDP)"
        if by == 0 or by <= horizon_a:
            return "endogenously optimized"
        return "unclear"

    return pd.DataFrame(
        {
            "delta_mw": delta,
            "length_km": b.length_km.fillna(a.length_km),
            "build_year": build_year,
            "bus0": b.bus0.fillna(a.bus0),
            "bus1": b.bus1.fillna(a.bus1),
            "country0": b.country0.fillna(a.country0),
            "country1": b.country1.fillna(a.country1),
            "classification": [classify(by, d) for by, d in zip(build_year, delta)],
        },
        index=idx,
    )


def summarize_diff(diff, scope=SCOPE):
    '''Expansion/reduction/net in TWkm (MW * km / 1e6), by classification.'''
    d = diff.loc[scope_mask(diff, scope) & (diff.classification != "no change")]
    twkm = lambda sub: (sub.delta_mw * sub.length_km).sum() / 1e6
    rows = {
        cls: {
            "n_branches": len(sub),
            "expansion_TWkm": twkm(sub[sub.delta_mw > 0]),
            "reduction_TWkm": twkm(sub[sub.delta_mw < 0]),
            "net_TWkm": twkm(sub),
        }
        for cls in ["planned (NEP/TYNDP)", "endogenously optimized", "unclear"]
        for sub in [d[d.classification == cls]]
    }
    return pd.DataFrame(rows).T


def changed_branches(diff, scope=SCOPE):
    '''Every branch with a nonzero capacity change, sorted by |delta| - the
    explicit per-line list behind the TWkm aggregate in `summarize_diff`.'''
    d = diff.loc[scope_mask(diff, scope) & (diff.classification != "no change")]
    d = d.reindex(d.delta_mw.abs().sort_values(ascending=False).index)
    return d[["delta_mw", "length_km", "build_year", "classification", "bus0", "bus1"]].round(1)

In [ ]:
def summarize_state(n, component, horizon, scope=SCOPE, label=""):
    '''Current-state snapshot: total capacity, TWkm, and the DE-internal vs.
    DE-crossborder split. The crossborder figure is the sum of branches'
    thermal/entry capacity, a simple proxy for transfer capacity - not the
    same thing as a regulatory NTC, which also accounts for loop flows and
    security margins and would need a flow-based study to compute properly.
    '''
    table = branch_table(n, component, horizon)
    t = table.loc[scope_mask(table, scope)]
    de_internal = t.country0.eq("DE") & t.country1.eq("DE")
    cross_border = t.country0.eq("DE") ^ t.country1.eq("DE")
    total_gw = t.capacity_mw.sum() / 1e3
    total_twkm = (t.capacity_mw * t.length_km).sum() / 1e6
    print(
        f"{label} {component}s (scope={scope}): {len(t)} branches, "
        f"{total_gw:.1f} GW total capacity, {total_twkm:.3f} TWkm total\n"
        f"  DE-internal:              {t[de_internal].capacity_mw.sum()/1e3:6.1f} GW "
        f"({de_internal.sum()} branches)\n"
        f"  DE cross-border capacity: {t[cross_border].capacity_mw.sum()/1e3:6.1f} GW "
        f"({cross_border.sum()} branches, thermal-capacity proxy, not a regulatory NTC)"
    )
    return t

In [ ]:
def check_names_in_canvas(names, canvas_table, label="candidate"):
    '''Validate that every branch name is present in the canvas (base) network -
    this is exactly what `build_grid_topology.py` requires at solve time; checking
    here catches a mismatch before burning a stochastic-optimization run on it.'''
    missing = pd.Index(names).difference(canvas_table.index)
    if len(missing):
        print(f"[{label}] MISSING from canvas ({len(missing)}): {list(missing)}")
    else:
        print(f"[{label}] OK - all {len(names)} name(s) found in canvas.")
    return missing

### Plotting

In [ ]:
EXTENT = {
    "whole_network": [-11, 30, 34, 65],
    "de_and_interconnectors": [-1, 24, 43, 60],
    "de_internal": [4, 16, 47, 56],
}


def _basemap(ax, scope):
    ax.add_feature(cfeature.OCEAN, facecolor="#d5e6f0", zorder=0)
    ax.add_feature(cfeature.LAND, facecolor="#f7f4ee", zorder=0)
    REGIONS.to_crs(epsg=4326).plot(ax=ax, transform=ccrs.PlateCarree(), facecolor="#eae6da",
                                    edgecolor="white", linewidth=0.5, zorder=0.5)
    ax.coastlines(resolution="50m", linewidth=0.3, color="gray", zorder=1)
    ax.set_extent(EXTENT[scope], crs=ccrs.PlateCarree())


def _draw_branches(ax, n, table, color_fn, width_fn):
    for name, row in table.iterrows():
        x = [n.buses.at[row.bus0, "x"], n.buses.at[row.bus1, "x"]]
        y = [n.buses.at[row.bus0, "y"], n.buses.at[row.bus1, "y"]]
        ax.plot(x, y, transform=ccrs.PlateCarree(), color=color_fn(row),
                linewidth=width_fn(row), zorder=2, solid_capstyle="round")


def _capacity_width(capacity_mw):
    return 0.3 + capacity_mw / WIDTH_SCALE_MW


def _capacity_legend(ax, sizes_gw=(5, 15, 30)):
    handles = [plt.Line2D([0], [0], color="steelblue", lw=_capacity_width(s * 1e3), label=f"{s} GW")
               for s in sizes_gw]
    ax.legend(handles=handles, loc="lower right", fontsize=7, framealpha=0.9, title="capacity")


def plot_networks_side_by_side(n_a, n_b, label_a, label_b, horizon_a, horizon_b, component="Line", scope=SCOPE):
    proj = ccrs.EqualEarth()
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), subplot_kw={"projection": proj})
    for ax, n, label, horizon in zip(axes, [n_a, n_b], [label_a, label_b], [horizon_a, horizon_b]):
        table = branch_table(n, component, horizon)
        table = table.loc[scope_mask(table, scope)]
        _basemap(ax, scope)
        _draw_branches(ax, n, table, lambda r: "steelblue", lambda r: _capacity_width(r.capacity_mw))
        buses = pd.Index(table[["bus0", "bus1"]].to_numpy().ravel()).unique()
        bx = n.buses.loc[buses]
        ax.scatter(bx.x, bx.y, transform=ccrs.PlateCarree(), s=6, color="firebrick", zorder=3)
        ax.set_title(f"{label}\n{component}s, scope={scope}")
        _capacity_legend(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
CLASS_COLORS = {
    "planned (NEP/TYNDP)": "seagreen",
    "endogenously optimized": "steelblue",
    "unclear": "goldenrod",
}


def plot_diff_map(n_ref, diff, label, component="Line", scope=SCOPE):
    table = diff.loc[scope_mask(diff, scope) & (diff.classification != "no change")]
    proj = ccrs.EqualEarth()
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": proj})
    _basemap(ax, scope)
    _draw_branches(
        ax, n_ref, table,
        lambda r: CLASS_COLORS[r.classification],
        lambda r: 0.5 + abs(r.delta_mw) / 1500,
    )
    ax.set_title(f"{label}\n{component} capacity diff, scope={scope}")
    handles = [plt.Line2D([0], [0], color=c, lw=2.5, label=k) for k, c in CLASS_COLORS.items()]
    ax.legend(handles=handles, loc="lower left", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()

In [ ]:
def export_scenario_csv(table, canvas_table, component, name, n_for_resolution, scope=SCOPE,
                         clusters=CLUSTERS, out_dir=REPO / "data/pypsa-de/grid_scenarios"):
    '''Write a `stochastic_grid_scenarios`-format CSV (name, s_nom|p_nom), scoped
    and validated against the canvas network `build_grid_topology.py` will apply
    it to. Branches outside `scope` are simply omitted from the CSV, so
    `build_grid_topology.py` leaves them at the canvas's own value - exactly the
    intent of scoping an export to "DE-relevant" branches only.'''
    nom = NOM_ATTR[component]
    table = table.loc[scope_mask(table, scope)]
    missing = check_names_in_canvas(table.index, canvas_table, label=name)
    if len(missing):
        raise ValueError(f"Refusing to export: {len(missing)} branch name(s) missing from canvas.")
    prefix = "lines" if component == "Line" else "links"
    resolution = temporal_resolution_label(n_for_resolution)
    path = Path(out_dir) / f"{prefix}_{name}_{scope}_{resolution}_{clusters}.csv"
    table[["capacity_mw"]].rename(columns={"capacity_mw": nom}).rename_axis("name").round(1).to_csv(path)
    print(f"wrote {path} ({len(table)} rows)")
    return path


def export_exogenous_scenario(target_year, canvas_lines, canvas_links, scope=SCOPE):
    '''The "specify a build year, get the exogenous-only (NEP/TYNDP/manual,
    zero endogenous optimization) topology" entry point: Lines + Links for one
    target year in a single call, built from `pristine` (the 2025 network) and
    exported as `exogenous{target_year}`.'''
    name = f"exogenous{target_year}"
    lines_path = export_scenario_csv(
        exogenous_scenario(pristine, "Line", target_year), canvas_lines, "Line", name, pristine, scope=scope
    )
    links_path = export_scenario_csv(
        exogenous_scenario(pristine, "Link", target_year), canvas_links, "Link", name, pristine, scope=scope
    )
    return lines_path, links_path

## Test run: `202600717-stoch-grid-testing-2030-70h`

Loading the four networks this analysis needs. `pre_2025` doubles as the
`pristine` reference for `exogenous_scenario`; `pre_2030` is `canvas` - the
same file `build_grid_topology.py` uses as the base network for the whole
stochastic-grid analysis.

In [ ]:
RESOURCES = REPO / "resources/202600717-stoch-grid-testing-2030-70h/KN2045_Mix/networks"
RESULTS = REPO / "results/202600717-stoch-grid-testing-2030-70h/KN2045_Mix/networks"

networks = {
    "pre_2025":  load_network(RESOURCES / "base_s_27__none_2025_final.nc"),
    "post_2025": load_network(RESULTS   / "base_s_27__none_2025.nc"),
    "pre_2030":  load_network(RESOURCES / "base_s_27__none_2030_final.nc"),
    "post_2030": load_network(RESULTS   / "base_s_27__none_2030.nc"),
}
networks["canvas"] = networks["pre_2030"]
pristine = networks["pre_2025"]

HORIZON = {"pre_2025": 2025, "post_2025": 2025, "pre_2030": 2030, "post_2030": 2030, "canvas": 2030}

## Current state (2025)

Baseline before looking at any diff: what the grid actually looks like in
`pre_2025`, in the same units (GW, TWkm) everything else below is reported in.

In [ ]:
_ = summarize_state(networks["pre_2025"], "Line", HORIZON["pre_2025"], label="2025 state")
_ = summarize_state(networks["pre_2025"], "Link", HORIZON["pre_2025"], label="2025 state")

## Lines (AC)

In [ ]:
plot_networks_side_by_side(networks["pre_2025"], networks["pre_2030"], "2025 state", "exogenous 2030 state (NEP/TYNDP only)",
                            HORIZON["pre_2025"], 2030, component="Line")
plot_networks_side_by_side(networks["pre_2030"], networks["post_2030"], "exogenous 2030 state", "optimal 2030 state",
                            HORIZON["pre_2030"], HORIZON["post_2030"], component="Line")

In [ ]:
lines_2025 = branch_table(networks["pre_2025"], "Line", HORIZON["pre_2025"])
lines_exog2030 = exogenous_scenario(pristine, "Line", 2030)
lines_opt2030 = branch_table(networks["post_2030"], "Line", HORIZON["post_2030"])

lines_2025_to_exog2030 = diff_table(lines_2025, lines_exog2030, HORIZON["pre_2025"], 2030)
lines_exog_to_optimal2030 = diff_table(lines_exog2030, lines_opt2030, 2030, HORIZON["post_2030"])

print("2025 state -> exogenous 2030 state (Lines):")
display(summarize_diff(lines_2025_to_exog2030))
print("changed branches:")
display(changed_branches(lines_2025_to_exog2030))

print("\nexogenous 2030 state -> optimal 2030 state (Lines):")
display(summarize_diff(lines_exog_to_optimal2030))
print("changed branches:")
display(changed_branches(lines_exog_to_optimal2030))

In [ ]:
plot_diff_map(networks["pre_2030"], lines_2025_to_exog2030, "2025 -> exogenous 2030", component="Line")
plot_diff_map(networks["post_2030"], lines_exog_to_optimal2030, "exogenous 2030 -> optimal 2030", component="Line")

## Links (DC)

In [ ]:
plot_networks_side_by_side(networks["pre_2025"], networks["pre_2030"], "2025 state", "exogenous 2030 state (NEP/TYNDP only)",
                            HORIZON["pre_2025"], 2030, component="Link")
plot_networks_side_by_side(networks["pre_2030"], networks["post_2030"], "exogenous 2030 state", "optimal 2030 state",
                            HORIZON["pre_2030"], HORIZON["post_2030"], component="Link")

In [ ]:
links_2025 = branch_table(networks["pre_2025"], "Link", HORIZON["pre_2025"])
links_exog2030 = exogenous_scenario(pristine, "Link", 2030)
links_opt2030 = branch_table(networks["post_2030"], "Link", HORIZON["post_2030"])

links_2025_to_exog2030 = diff_table(links_2025, links_exog2030, HORIZON["pre_2025"], 2030)
links_exog_to_optimal2030 = diff_table(links_exog2030, links_opt2030, 2030, HORIZON["post_2030"])

print("2025 state -> exogenous 2030 state (Links):")
display(summarize_diff(links_2025_to_exog2030))
print("changed branches:")
display(changed_branches(links_2025_to_exog2030))

print("\nexogenous 2030 state -> optimal 2030 state (Links):")
display(summarize_diff(links_exog_to_optimal2030))
print("changed branches:")
display(changed_branches(links_exog_to_optimal2030))

In [ ]:
plot_diff_map(networks["pre_2030"], links_2025_to_exog2030, "2025 -> exogenous 2030", component="Link")
plot_diff_map(networks["post_2030"], links_exog_to_optimal2030, "exogenous 2030 -> optimal 2030", component="Link")

## Export scenario CSVs

Deterministic grid realizations to feed `stochastic_grid_scenarios.scenarios`,
replacing the arbitrary `reduced_10`/`reduced_40` cuts:

- **`2025state`**: the grid stays essentially where it was in 2025 (a
  stagnation scenario). Note this is *exactly* `exogenous_scenario(pristine,
  component, 2025)` - "2025 state" already includes any project whose
  `build_year <= 2025`, i.e. anything committed by 2025 is "exogenous 2025",
  same build-year gating as every other year.
- **`optimal2025`**: the model's own cost-optimal 2025 build-out (from
  `post_2025`, the solved 2025 network) - the "2025" counterpart to
  `optimal2030`, i.e. what if the grid ends up exactly where the 2025 solve's
  own endogenous choices left it, with no further expansion by 2030. Every
  single branch (44/44 lines, 104/104 DC links) differs from `2025state` here,
  so this is not a redundant pairing - `post_2025` wasn't exported earlier,
  which was a gap, not a "diff is zero" situation.
- **`exogenous{year}`**: only committed NEP/TYNDP/manual projects built by
  `year` - no endogenous expansion. `export_exogenous_scenario(year, ...)` is
  the general entry point for this - pick any year, not just 2030; called
  below for both 2030 and 2040 to demonstrate.
- **`optimal2030`**: the model's own cost-optimal 2030 build-out.

All validated against `canvas` (`pre_2030`) before writing, scoped to
`SCOPE`, and named with the scope and temporal resolution baked in.

In [ ]:
canvas_lines = branch_table(networks["canvas"], "Line", HORIZON["canvas"])
canvas_links = branch_table(networks["canvas"], "Link", HORIZON["canvas"])

lines_opt2025 = branch_table(networks["post_2025"], "Line", HORIZON["post_2025"])
links_opt2025 = branch_table(networks["post_2025"], "Link", HORIZON["post_2025"])

export_scenario_csv(lines_2025, canvas_lines, "Line", "2025state", networks["pre_2025"])
export_scenario_csv(links_2025, canvas_links, "Link", "2025state", networks["pre_2025"])

export_scenario_csv(lines_opt2025, canvas_lines, "Line", "optimal2025", networks["post_2025"])
export_scenario_csv(links_opt2025, canvas_links, "Link", "optimal2025", networks["post_2025"])

export_exogenous_scenario(2030, canvas_lines, canvas_links)
export_exogenous_scenario(2040, canvas_lines, canvas_links)  # any year works - demonstrating it's not hardcoded to 2030

export_scenario_csv(lines_opt2030, canvas_lines, "Line", "optimal2030", networks["post_2030"])
export_scenario_csv(links_opt2030, canvas_links, "Link", "optimal2030", networks["post_2030"])

### Sanity check: validate an *existing* candidate CSV against a chosen canvas

Demonstrates the standalone validator on the existing `reduced_40` scenario
CSVs against the 2030 canvas - this is the check that would otherwise only
surface as a `ValueError` deep inside `build_grid_topology.py` at solve time.

In [ ]:
candidate_lines = pd.read_csv(REPO / f"data/pypsa-de/grid_scenarios/lines_reduced_40_{CLUSTERS}.csv", index_col=0)
candidate_lines.index = candidate_lines.index.astype(str)
check_names_in_canvas(candidate_lines.index, canvas_lines, label="lines_reduced_40")

candidate_links = pd.read_csv(REPO / f"data/pypsa-de/grid_scenarios/links_reduced_40_{CLUSTERS}.csv", index_col=0)
candidate_links.index = candidate_links.index.astype(str)
check_names_in_canvas(candidate_links.index, canvas_links, label="links_reduced_40")

## Does this file selection make sense?

Mostly yes, with one correction that turned out to matter more than expected:

- **"the network used to construct the stochastic network" is the 2030
  prenetwork** - literal same file (`base_s_27__none_2030_final.nc`). Not a
  fifth distinct state.
- **The "planned 2030" state was wrong in the previous version of this
  notebook, and the fix changed the numbers, not just the framing.** Reading
  `2030_final`'s own `p_nom`/`s_nom` at face value silently includes whatever
  the 2025 solve endogenously decided to build, since brownfield carryover
  overwrites "existing" (`build_year==0`) capacity in place. The corrected
  `exogenous_scenario` builds strictly from `2025_final` (the pristine, never
  -solved first horizon) instead, gating every branch by its own `build_year`.
  This is exactly the check worth re-running whenever this notebook is pointed
  at a different run: confirm the *first* configured planning horizon's
  prenetwork is genuinely never brownfield-updated before treating it as
  "pristine" - if `stochastic_grid_scenarios.planning_horizons` or
  `scenario.planning_horizons` changes, so does which file plays that role.
- **Why the staged comparison is worth keeping**: only `2025 -> exogenous2030
  -> optimal2030` cleanly separates "a committed project came online" from
  "the solver endogenously expanded the grid" - collapsing the two steps mixes
  both effects into one undifferentiable number.
- **By construction**, `exogenous 2030 -> optimal 2030` is always 100%
  "endogenously optimized" for the *same* target year (any committed project
  by 2030 is already in `exogenous2030`) - the "planned" bucket only does
  meaningful work in the `2025 -> exogenous2030` step.